# Querying CTAO SDC ObsCore metadata through TAP

This notebook demonstrates how to query the CTAO Science Data Challenge ObsCore table directly from Python using the Virtual Observatory TAP protocol.

TAP service:

`http://voparis-tap-he.obspm.fr/tap`

ObsCore table:

`ctao_sdc.obscore`

In [1]:
from textwrap import dedent

import pandas as pd
import pyvo as vo

TAP_URL = "http://voparis-tap-he.obspm.fr/tap"
OBSCORE_TABLE = "ctao_sdc.obscore"

service = vo.dal.TAPService(TAP_URL)

In [2]:
query = dedent(f"""
    SELECT TOP 10
        obs_id,
        dataproduct_type,
        calib_level,
        target_name,
        s_ra,
        s_dec,
        t_min,
        t_max,
        em_min,
        em_max,
        access_url
    FROM {OBSCORE_TABLE}
""")

results = service.search(query)
table = results.to_table()
df = table.to_pandas()

df.head()

,obs_id,dataproduct_type,calib_level,target_name,s_ra,s_dec,t_min,t_max,em_min,em_max,access_url
0,5000000396,event-list,2,KSP: Transients-GRB - Event #355,107.177099,-3.099358,61797.110564,61797.168377,6.213929e-21,9.848414e-17,http://voparis-tap-he.obspm.fr/ctao_sdc/q/data...
1,5000000379,event-list,2,EGal Survey,216.770682,18.639032,61796.258785,61796.272673,6.213929e-21,9.848414e-17,http://voparis-tap-he.obspm.fr/ctao_sdc/q/data...
2,5000000400,event-list,2,AGN-Flares-External - 3C 279,194.750115,-5.788880,61797.179991,61797.185778,6.213929e-21,9.848414e-17,http://voparis-tap-he.obspm.fr/ctao_sdc/q/data...
3,5000000050,event-list,2,AGN-LTM - 1ES 1426+428,218.087902,42.668573,61773.266262,61773.271470,6.213929e-21,9.848414e-17,http://voparis-tap-he.obspm.fr/ctao_sdc/q/data...
4,5000000545,event-list,2,EGal Survey,199.126962,20.474524,61801.046042,61801.091701,6.213929e-21,9.848414e-17,http://voparis-tap-he.obspm.fr/ctao_sdc/q/data...


The result is an Astropy table converted to a Pandas DataFrame. Each row corresponds to one dataset or observation product described by ObsCore metadata.

In [3]:
metadata_query = dedent(f"""
    SELECT TOP 57
        column_name,
        datatype,
        description,
        unit,
        ucd
    FROM TAP_SCHEMA.columns
    WHERE table_name = '{OBSCORE_TABLE}'
    ORDER BY column_name
""")

columns = service.search(metadata_query).to_table().to_pandas()
columns.head(57)

,column_name,datatype,description,unit,ucd
0,access_estsize,long,Estimated size of data product,kbyte,phys.size;meta.file
1,access_format,char,MIME type of the resource at access_url,,meta.code.mime
2,access_url,char,The URL at which to obtain the data set.,,meta.ref.url
3,access_url_this,char,URL for direct download of the event-list,,meta.ref.url
4,analysis_mode,char,Data reduction/analysis mode,,meta.code;obs.param
5,calib_level,short,Amount of data processing that has been applie...,,meta.code;obs.calib
6,coverage,char,"introduces an ascii (ST)MOC, v2 (2D footprint ...",,pos.outline;obs.field
7,datalink,char,URL of a datalink document for this dataset,,meta.ref.url
8,dataproduct_subtype,char,Data product specific type,,meta.code.class
9,dataproduct_type,char,High level scientific classification of the da...,,meta.code.class


In [4]:
# M82
ra = 148.968458
dec = 69.679703
radius_deg = 5.0

region_query = dedent(f"""
    SELECT TOP 50
        obs_id,
        target_name,
        s_ra,
        s_dec,
        t_min,
        t_max,
        em_min,
        em_max
    FROM {OBSCORE_TABLE}
    WHERE 1 = CONTAINS(
        POINT('ICRS', s_ra, s_dec),
        CIRCLE('ICRS', {ra}, {dec}, {radius_deg})
    )
""")

region_df = service.search(region_query).to_table().to_pandas()
region_df.head()

,obs_id,target_name,s_ra,s_dec,t_min,t_max,em_min,em_max
0,5000000007,SFS - M82,150.983459,69.668154,61771.164086,61771.219641,6.213929e-21,9.848414e-17
1,5000000174,SFS - M82,148.968455,68.979698,61789.010521,61789.013935,6.213929e-21,9.848414e-17
2,5000000070,SFS - M82,148.968455,68.979698,61775.093738,61775.131875,6.213929e-21,9.848414e-17
3,5000000162,SFS - M82,148.968455,70.379698,61788.006551,61788.009965,6.213929e-21,9.848414e-17
4,5000000163,SFS - M82,148.968455,68.979698,61788.010023,61788.013437,6.213929e-21,9.848414e-17
